This script loads the generated pickle file and checks the prediction for an input stay_id

In [3]:
import pandas as pd
import numpy as np
import pickle

# Load the saved model and other objects
with open('final_readmission_model.pkl', 'rb') as f:
    loaded_objects = pickle.load(f)

# Extract objects
loaded_best_rf_model = loaded_objects['best_rf_model']
loaded_top_features = loaded_objects['top_features']
loaded_imputer = loaded_objects['imputer']
loaded_encoder = loaded_objects['encoder']
loaded_scaler = loaded_objects['scaler']
loaded_scaler_final = loaded_objects['scaler_final']
merge_final = loaded_objects['merge_final']

# Define the prediction function
def predict_readmission(stay_id, merge_final, best_rf_model, top_features, imputer, encoder, scaler, scaler_final):
    # Extract the data for the given stay_id
    print(f"Predicting readmission likelihood for stay_id: {stay_id}")
    stay_data = merge_final[merge_final['stay_id'] == stay_id]

    # Check if data exists for the given stay_id
    if stay_data.empty:
        print(f"No data found for stay_id: {stay_id}")
        return None, None, None, None

    # Drop unnecessary columns for prediction
    stay_data = stay_data.drop(columns=['subject_id', 'readmitted'], errors='ignore')

    # Prepare the data for prediction
    X = stay_data

    # Identify numerical and categorical columns
    numerical_columns = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
    categorical_columns = X.select_dtypes(include=['object']).columns.tolist()

    # Impute missing values for numerical columns
    X_imputed = imputer.transform(X[numerical_columns])
    X_imputed_df = pd.DataFrame(X_imputed, columns=numerical_columns)

    # Encode categorical columns
    X_encoded = encoder.transform(X[categorical_columns])
    X_encoded_df = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out(categorical_columns))

    # Standardize numerical features
    X_scaled = scaler.transform(X_imputed_df)
    X_scaled_df = pd.DataFrame(X_scaled, columns=X_imputed_df.columns)

    # Combine scaled and encoded features
    X_processed = pd.concat([X_scaled_df, X_encoded_df], axis=1)

    # Select top features
    X_final = X_processed[top_features]

    # Standardize the final selected features
    X_final_scaled = scaler_final.transform(X_final)

    # Predict the readmission likelihood
    readmission_prob = best_rf_model.predict_proba(X_final_scaled)[:, 1]

    # Get the ICU admission time and the last abs_event_time from the stay_data
    icu_intime = stay_data['icu_intime'].values[0] if not stay_data.empty else None
    last_abs_event_time = stay_data['abs_event_time'].max() if not stay_data.empty else None

    # Convert numpy.datetime64 to Python datetime and format the times
    icu_intime = pd.to_datetime(icu_intime) if icu_intime is not None else None
    last_abs_event_time = pd.to_datetime(last_abs_event_time) if last_abs_event_time is not None else None

    icu_intime_str = icu_intime.strftime('%Y-%m-%d %H:%M:%S') if icu_intime is not None else None
    last_abs_event_time_str = last_abs_event_time.strftime('%Y-%m-%d %H:%M:%S') if last_abs_event_time is not None else None

    # Calculate the number of days between icu_intime and last_abs_event_time
    if icu_intime is not None and last_abs_event_time is not None:
        delta = last_abs_event_time - icu_intime
        days_between = delta.days + delta.seconds / (3600 * 24)  # Add fraction of days
    else:
        days_between = None

    return icu_intime_str, last_abs_event_time_str, readmission_prob[0], days_between

# Example usage
if __name__ == "__main__":
    stay_id_to_predict = int(input("Enter the stay_id: "))
    icu_intime, last_abs_event_time, readmission_likelihood, days_between = predict_readmission(
        stay_id_to_predict, 
        merge_final, 
        loaded_best_rf_model, 
        loaded_top_features, 
        loaded_imputer, 
        loaded_encoder, 
        loaded_scaler, 
        loaded_scaler_final
    )

    if icu_intime is not None:
        print(f"ICU Intime for stay_id {stay_id_to_predict}: {icu_intime}")
        print(f"Days from ICU Intime to the last abs_event_time: {days_between:.2f}")
        readmission_likelihood_percentage = readmission_likelihood * 100
        print(f"Readmission likelihood percentage for stay_id {stay_id_to_predict} at the last event time: {readmission_likelihood_percentage:.2f}")
    else:
        print(f"No data found for stay_id: {stay_id_to_predict}")


Enter the stay_id: 31248398
Predicting readmission likelihood for stay_id: 31248398
ICU Intime for stay_id 31248398: 2201-07-07 19:40:00
Days from ICU Intime to the last abs_event_time: 0.75
Readmission likelihood percentage for stay_id 31248398 at the last event time: 98.17
